In [6]:
# Used for command-line argument parsing, allowing you to pass arguments (like the query) when running the script from the command line.
import argparse

# from dataclasses import dataclass
'''
Without @dataclass, you would have to manually define the __init__ method to initialize class attributes. With @dataclass, it’s done automatically.

from dataclasses import dataclass

@dataclass
class Book:
    title: str
    author: str
    year: int

# Automatically adds an __init__ method:
# def __init__(self, title, author, year):
#     self.title = title
#     self.author = author
#     self.year = year

book = Book("1984", "George Orwell", 1949)
print(book)  # Book(title='1984', author='George Orwell', year=1949)
'''

# A vector database that stores the document embeddings and allows efficient similarity searches.
from langchain_community.vectorstores import Chroma

# Convert the text into vector representations
from langchain_openai import OpenAIEmbeddings

# Generate responses to queries based on a given prompt
from langchain_openai import ChatOpenAI

# A utility class to define a template structure for the chat prompt, allowing dynamic insertion of context and queries.
from langchain.prompts import ChatPromptTemplate

In [7]:
# A string variable that sets the directory where the Chroma database will be stored. The database stores the vectorized documents.
CHROMA_PATH = "chroma"

# A simple template for generating chat-based queries. The template expects a {context} (retrieved from Chroma database) and a {question} (the user’s query).
# •	The context will be inserted dynamically, and the question is the query asked by the user.
# •	The structure provides the context first, followed by the question, making sure the answer is based only on the retrieved context.
PROMPT_TEMPLATE = """
Answer the question based only on the following context:

{context}

---

Answer the question based on the above context: {question}
"""

In [ ]:
def main():
    
    # Create CLI.
    # The command-line interface (CLI) allows the user to provide a query (query_text) when running the script. 
    # This enables dynamic input when executing the script, so it can respond to different queries each time.
    parser = argparse.ArgumentParser()
    
    # The add_argument method is used to define the arguments that the script can accept.
    # This argument will not have an actual value yet at this point in the script. It’s simply setting up a placeholder for when you run the program from the command line.
    # argument: This is the name of the argument, and it acts as the place-holder for what you will pass when running the script.
    # type: Specifies that the value passed for query_text must be a string.
    # help: A short description of this argument, which will show up when you run python script.py --help.
    # Example: Running the script in the terminal with python script.py "What is NLP?" would set the query text to "What is NLP?".
    parser.add_argument("query_text", type = str, help = "What is your question?")
    
    # Parse the arguments passed in the command-line.
    # The argparse.ArgumentParser() object (which is stored in the parser variable) defines the command-line arguments that the script expects (in this case, query_text).
    # parse_args() processes the command-line input and stores the parsed arguments in the args variable.
    # This method reads the command-line arguments passed when running the script and organizes them in a structured format, typically as an object with attributes corresponding to each argument.
    args = parser.parse_args()
    '''
    • Here, script.py is being run with "What is the capital of France?" as an argument for query_text.
    python script.py "What is the capital of France?"

    • After calling parse_args(), args would look like this:
    args.query_text = "What is the capital of France?"
    '''
    
    # This line assigns the value of the parsed query_text argument (from args) to the variable query_text.
    query_text = args.query_text

    # Initialize the OpenAIEmbeddings class with the text-embedding-3-large model.
    embedding_function = OpenAIEmbeddings(
        model = "text-embedding-ada-002",
        #model = "text-embedding-3-large",
    )
    
    # Initializes the Chroma database with the provided embedding function. 
    # The database is created from document embeddings, and it’s stored at the directory specified by CHROMA_PATH.
    db = Chroma(persist_directory = CHROMA_PATH, embedding_function = embedding_function)

    # Performs a similarity search in the Chroma database for the top 3 (k = 3) results that are most relevant to the input query_text.
    # k = 3: This limits the number of results to 3. 
    # It uses the embedding-based similarity to match the query with the document chunks.
    results = db.similarity_search_with_relevance_scores(query_text, k = 3)
    
    # If the results list is empty or the first result’s relevance score is lower than 0.7, it will print a message and exit the function.
    if len(results) == 0 or results[0][1] < 0.7:
        print(f"Unable to find matching results.")
        return

    # Concatenates the relevant documents(data) from the search results into a single string, which will be used as context for generating the answer.
    # •	results: a list of results returned by the similarity search from the Chroma database. Each element in results is a tuple: (document, score), where document is the retrieved document, and score is the relevance score of that document to the query.
	# •	doc.page_content: extracts the actual content of each document (i.e., the text from the document itself).
	# •	join(): combines the page_content from multiple documents into a single string. 
    # •	"---": The documents are separated by "---" and 2 new rows(\n\n) to make it easier to differentiate between them when forming the prompt.
    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    '''
    # Initialize an empty list to store document contents.
    context_list = []

    # Iterate over the search results (which contain tuples of (document, relevance score)).
    for doc, _score in results:
        # Extract the page content from the document and append it to the context_list.
        context_list.append(doc.page_content)

    # Now join all the content together with the separator "\n\n---\n\n" between documents.
    context_text = "\n\n---\n\n".join(context_list)
    '''
    
    # Creates a prompt template from the PROMPT_TEMPLATE string. 
    # The PROMPT_TEMPLATE defines how the prompt should be structured, and the placeholders {context} and {question} will be replaced dynamically.
    prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    prompt = prompt_template.format(context = context_text, question = query_text)
    
    print(prompt)

    # This initializes an instance of OpenAI’s chat model (likely GPT-3 or GPT-4). 
    # It is used to generate human-like responses based on the input prompt.
    model = ChatOpenAI(
        model = "gpt-3.5-turbo",
        # model="gpt-4o-mini"
    )
    # model.invoke(prompt): This line sends the formatted prompt to the model and receives the model’s generated response.
    # The model uses the context (documents) and the question to generate an answer.
    response_text = model.invoke(prompt)
    
    # Since response is an AIMessage object, access the 'content' directly so that metadata can be extracted from the response.
    # response.content: This line extracts the content of the response from the AIMessage object.
    # Your code works with .content because ChatOpenAI.invoke(prompt) returns an AIMessage object. This object contains the model’s response as an attribute called .content.
    response_text = response_text.content  # Access the content from the response object
    '''
    Without response_text = response_text.content)
    Response: content='Based on the context provided, the Mad Hatter is described as funny, muttering, and possibly raving mad.' response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 203, 'total_tokens': 228, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-5684ff13-311d-44db-b6f0-859ce0a49c4b-0' usage_metadata={'input_tokens': 203, 'output_tokens': 25, 'total_tokens': 228}
    Sources: ['data/books/alice_in_wonderland.md', 'data/books/alice_in_wonderland.md', 'data/books/alice_in_wonderland.md']
    
    With response_text = response_text.content)
    Response: Based on the context provided, the Mad Hatter is described as having a "funny watch" and muttering a response to Alice's comment about the watch. There is no specific physical description of the Mad Hatter's appearance given in this context.
    Sources: ['data/books/alice_in_wonderland.md', 'data/books/alice_in_wonderland.md', 'data/books/alice_in_wonderland.md']
    '''

    # doc.metadata.get("source", None): For each document in the results, this line tries to extract the source metadata (such as the document’s name or URL).
	# If no "source" is found in the document’s metadata, it returns None.
	# The sources list will contain all the sources of the documents that contributed to the generated response.
    sources = [doc.metadata.get("source", None) for doc, _score in results]
    
    # f"Response: {response_text}\nSources: {sources}": This formats the final response. 
    # It includes both the generated response (response_text) and the sources (documents) that contributed to the answer.
    formatted_response = f"Response: {response_text}\nSources: {sources}"
    
    print(formatted_response)

In [ ]:
# To ensure that the script runs only when executed directly (not when imported as a module)
if __name__ == "__main__":
    main()

In [13]:
db = Chroma(persist_directory = CHROMA_PATH, embedding_function = get_embedding_function())

In [12]:
from langchain_community.embeddings.ollama import OllamaEmbeddings
def get_embedding_function():
    
    #embeddings = BedrockEmbeddings(
    #    credentials_profile_name = "default", 
    #    region_name = "us-east-1"
    #)
    
    embeddings = OllamaEmbeddings(model = "nomic-embed-text")
    
    return embeddings

In [23]:
db.get()

{'ids': ['0038a4f3-f0a6-4e80-a2a2-ccf46bbfe672',
  '008222d0-c189-4237-8ae8-c34fdadb5e74',
  '0086e38c-7332-491c-be24-04bb4e458858',
  '00fb3a54-f525-47ed-bb8d-23815c79e752',
  '01b0692e-604e-4d08-8fd1-86fbc070573e',
  '01bf6968-49a8-47bc-a760-78d4dcbaea36',
  '022a9f03-04e1-4d3f-937c-aa3d19795b64',
  '02ce70a7-f74b-4be5-baad-8dd2a88cc195',
  '030da453-0c44-430d-98c6-a716de7e9c55',
  '0340aa2d-baa3-48e5-8001-d2b755bfc08f',
  '03c891a6-1a87-4bf9-a1bf-6fa5f06b674b',
  '03f3661c-0264-41bd-b521-7e9356b5faa3',
  '05a24dad-f7f4-4181-9eb2-bcb29c976dcd',
  '0633ea8d-6e52-497d-ad66-28313166e32e',
  '0642c420-5e5c-4174-82db-a3a170f5b3dd',
  '0646e7c6-1d26-489b-ab47-6e1d46dda899',
  '067968eb-f792-4d82-9a88-d6098d3a1e1f',
  '077e4901-baf5-49a5-a1d5-93afa7cb43f3',
  '080f3b1d-3e00-44fb-98a3-d78ecd405b04',
  '083a43c6-7d34-4d64-94a7-f4ff76959d38',
  '0856b304-e04a-4c07-aff9-ef8e997b3e45',
  '08efbc57-8c03-4566-96a1-e05647396095',
  '0a9cddca-8b65-4bb8-a01e-b3268cc113e8',
  '0ab8f41f-b366-482c-8404-